# 02 — Train CamemBERT Classifier

**Input:** `data/questions_dataset/train.csv`, `val.csv`, `test.csv` (from notebook 01)

**Output:** `models/camembert_classifier/` — fine-tuned model + tokenizer

---

## Pipeline steps
1. Load and tokenize datasets
2. Configure CamemBERT for sequence classification (5 classes)
3. Train with early stopping on validation F1
4. Evaluate on test set — accuracy, precision, recall, F1-macro, confusion matrix
5. Analyse misclassifications
6. Save model and tokenizer

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.append('..')

DATASET_DIR = Path('../data/questions_dataset')
MODEL_DIR   = Path('../models/camembert_classifier')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LABELS = [
    'information_generale',
    'sensible',
    'avis_medical_personnalise',
    'urgence',
    'hors_perimetre',
]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

## Step 1 — Load and tokenize datasets

In [ ]:
from transformers import CamembertTokenizer
from datasets import Dataset

tokenizer = CamembertTokenizer.from_pretrained('camembert-base')

# train_df = pd.read_csv(DATASET_DIR / 'train.csv')
# val_df   = pd.read_csv(DATASET_DIR / 'val.csv')
# test_df  = pd.read_csv(DATASET_DIR / 'test.csv')

# def encode(df):
#     df = df.copy()
#     df['label'] = df['label'].map(LABEL2ID)
#     dataset = Dataset.from_pandas(df)
#     return dataset.map(
#         lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
#         batched=True
#     )

# train_ds = encode(train_df)
# val_ds   = encode(val_df)
# test_ds  = encode(test_df)
# print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## Step 2 — Configure CamemBERT for sequence classification

In [ ]:
from transformers import CamembertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

# model = CamembertForSequenceClassification.from_pretrained(
#     'camembert-base',
#     num_labels=len(LABELS),
#     id2label=ID2LABEL,
#     label2id=LABEL2ID,
# )

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=-1)
#     return {
#         'accuracy': accuracy_score(labels, preds),
#         'f1_macro': f1_score(labels, preds, average='macro'),
#     }

## Step 3 — Train with early stopping on validation F1

In [ ]:
# training_args = TrainingArguments(
#     output_dir=str(MODEL_DIR),
#     num_train_epochs=10,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     evaluation_strategy='epoch',
#     save_strategy='epoch',
#     load_best_model_at_end=True,
#     metric_for_best_model='f1_macro',
#     greater_is_better=True,
#     logging_steps=10,
#     seed=42,
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     compute_metrics=compute_metrics,
# )

# trainer.train()

## Step 4 — Evaluate on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# results = trainer.predict(test_ds)
# preds   = np.argmax(results.predictions, axis=-1)
# labels  = results.label_ids

# print(classification_report(labels, preds, target_names=LABELS))

# cm = confusion_matrix(labels, preds)
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', xticklabels=LABELS, yticklabels=LABELS, cmap='Blues')
# plt.title('Confusion Matrix — Test Set')
# plt.ylabel('True Label')
# plt.xlabel('Predicted Label')
# plt.tight_layout()
# plt.show()

## Step 5 — Analyse misclassifications

**Critical check:** any `avis_medical_personnalise` or `urgence` classified as `information_generale` is a safety failure — must be zero.

In [ ]:
# test_df_reset = test_df.reset_index(drop=True)
# test_df_reset['predicted'] = [ID2LABEL[p] for p in preds]
# test_df_reset['true']      = [ID2LABEL[l] for l in labels]

# errors = test_df_reset[test_df_reset['predicted'] != test_df_reset['true']]
# print(f'Total misclassifications: {len(errors)}')

# SAFETY CHECK — dangerous misclassifications
# dangerous = errors[
#     errors['true'].isin(['avis_medical_personnalise', 'urgence']) &
#     (errors['predicted'] == 'information_generale')
# ]
# print(f'DANGEROUS misclassifications (must be 0): {len(dangerous)}')
# if len(dangerous) > 0:
#     print('WARNING: retrain or adjust confidence threshold before deployment')
#     display(dangerous)

## Step 6 — Save model and tokenizer

In [ ]:
# model.save_pretrained(MODEL_DIR)
# tokenizer.save_pretrained(MODEL_DIR)

# Save label map for runtime loading
# with open(MODEL_DIR / 'label_map.json', 'w') as f:
#     json.dump({'id2label': ID2LABEL, 'label2id': LABEL2ID}, f)

# print(f'Model saved to {MODEL_DIR}')